# Lab 2 — Forward Pass and Error

<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Task 2</strong></h4>
<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
    <b>Instruction:</b> Perform a single forward pass and compute for the error.
</p>

$$x = \begin{bmatrix} 1 \\ 0 \\ 1 \end{bmatrix}$$

$$y = \begin{bmatrix} 1 \end{bmatrix}$$

$$f=max(0, Z_n)$$

$$\text{hidden unit weights} =
\begin{bmatrix}
w_{11} = 0.2 && w_{12} = -0.3 \\
w_{13} = 0.4 && w_{14} = 0.1 \\
w_{15} = -0.5 && w_{16} = 0.2
\end{bmatrix}$$

$$\text{output unit weights} =
\begin{bmatrix}
w_{21} = -0.3 \\
w_{22} = -0.2
\end{bmatrix}$$

$$\theta =
\begin{bmatrix}
\theta_{1} = -0.4 \\
\theta_{2} = 0.2 \\
\theta_{3} = 0.1
\end{bmatrix}$$
</div>

### My thought process

Before I jump into code, I want to make sure I actually understand what network this table of weights is describing, because the notation $w_{11}, w_{12}, \dots$ doesn't tell me the architecture on its own — I have to read the layout of the matrix first.

**Reading the weight table.** The *hidden unit weights* matrix has 3 rows and 2 columns. I'm treating the 3 rows as lining up with my 3 inputs $x_1, x_2, x_3$, and the 2 columns as lining up with my 2 hidden units, $H_1$ and $H_2$. So my interpretation is:

- $H_1$ gets $w_{11}$ (from $x_1$), $w_{13}$ (from $x_2$), $w_{15}$ (from $x_3$)
- $H_2$ gets $w_{12}$ (from $x_1$), $w_{14}$ (from $x_2$), $w_{16}$ (from $x_3$)

The *output unit weights* are just one column with 2 entries, $w_{21}$ and $w_{22}$ — one weight per hidden unit, both feeding into the single output neuron $O$.

And $\theta_1, \theta_2, \theta_3$ are the **biases** for $H_1$, $H_2$, and $O$ respectively. I'll use the usual convention for how the bias gets added in:

$$Z_n = \left(\sum_i w_i \cdot x_i\right) + \theta_n$$

So putting it together, the network I need to simulate is: **3 inputs → 2 hidden units (ReLU) → 1 output unit (ReLU)**.

**Activation function.** $f = \max(0, Z_n)$ is just **ReLU**, applied to whatever weighted sum $Z_n$ I'm computing at that point (I use it for both the hidden layer and the output layer here).

**My plan for the forward pass:**
1. Compute the net input $Z$ for each hidden unit, then apply ReLU to get $H_1, H_2$.
2. Use $H_1, H_2$ to compute the net input for the output unit, then apply ReLU to get $\hat{y}$ (my prediction).
3. Compare $\hat{y}$ against the true label $y = 1$ to get the error.


In [1]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

# ---- values given to me in the problem ----
x = np.array([1, 0, 1], dtype=float)   # inputs x1, x2, x3
y = np.array([1.0])                    # true target

# hidden layer weights: rows = x1,x2,x3 | columns = H1, H2
W1 = np.array([
    [ 0.2, -0.3],   # w11, w12  (from x1)
    [ 0.4,  0.1],   # w13, w14  (from x2)
    [-0.5,  0.2],   # w15, w16  (from x3)
])
theta_hidden = np.array([-0.4, 0.2])   # theta1 (H1), theta2 (H2)

# output layer weights: from H1, H2 -> O
W2 = np.array([-0.3, -0.2])
theta_output = np.array([0.1])         # theta3

def relu(z):
    return np.maximum(0, z)

print("x =", x)
print("y =", y)

x = [1. 0. 1.]
y = [1.]


### Step 1 — Hidden layer

Here's what I need to compute for each hidden unit:

$$Z_{H_1} = w_{11}x_1 + w_{13}x_2 + w_{15}x_3 + \theta_1, \qquad H_1 = \max(0, Z_{H_1})$$
$$Z_{H_2} = w_{12}x_1 + w_{14}x_2 + w_{16}x_3 + \theta_2, \qquad H_2 = \max(0, Z_{H_2})$$


In [2]:
Z_hidden = x @ W1 + theta_hidden   # net input for H1 and H2, both at once
H = relu(Z_hidden)                 # apply ReLU

print("Z_hidden (Z_H1, Z_H2):", Z_hidden)
print("H (H1, H2) after ReLU  :", H)

Z_hidden (Z_H1, Z_H2): [-0.7  0.1]
H (H1, H2) after ReLU  : [0.  0.1]


When I check this by hand: $Z_{H_1} = 0.2(1) + 0.4(0) + (-0.5)(1) + (-0.4) = -0.7$, which is **negative**, so ReLU clips $H_1$ down to $0$. Meanwhile $Z_{H_2} = -0.3(1) + 0.1(0) + 0.2(1) + 0.2 = 0.1$, which is positive, so $H_2 = 0.1$ just passes through unchanged. That matches what the code printed above, so I'm confident I set this up correctly — and it's a nice reminder of what ReLU actually does: it zeroes out any negative net input and lets positive net input through as-is.

### Step 2 — Output layer

Now I take $H_1, H_2$ and push them through the output unit the same way:

$$Z_O = w_{21}H_1 + w_{22}H_2 + \theta_3, \qquad \hat{y} = O = \max(0, Z_O)$$


In [3]:
Z_output = H @ W2 + theta_output
y_hat = relu(Z_output)

print("Z_output:", Z_output)
print("y_hat (predicted output):", y_hat)

Z_output: [0.08]
y_hat (predicted output): [0.08]


### Step 3 — Compute the error

I got $\hat{y} = 0.08$, but the true target is $y = 1$. The most direct error signal is just the difference:

$$\text{Error} = y - \hat{y}$$

Since my output here is a ReLU value rather than a probability, I think it makes more sense to treat this like a regression problem, so I'll also report the **squared error** — this is the quantity I'd actually be minimizing if I were training this network:

$$E = \frac{1}{2}(y - \hat{y})^2$$


In [4]:
error = y - y_hat
squared_error = 0.5 * (y - y_hat) ** 2

print(f"Predicted output (y_hat) : {y_hat[0]:.4f}")
print(f"True label (y)           : {y[0]:.4f}")
print(f"Error (y - y_hat)        : {error[0]:.4f}")
print(f"Squared error 1/2(y-yhat)^2 : {squared_error[0]:.4f}")

Predicted output (y_hat) : 0.0800
True label (y)           : 1.0000
Error (y - y_hat)        : 0.9200
Squared error 1/2(y-yhat)^2 : 0.4232


### What I found

| Quantity | Value |
|---|---|
| $Z_{H_1}$ | $-0.70$ |
| $H_1 = \max(0, Z_{H_1})$ | $0.00$ |
| $Z_{H_2}$ | $0.10$ |
| $H_2 = \max(0, Z_{H_2})$ | $0.10$ |
| $Z_O$ | $0.08$ |
| $\hat{y} = \max(0, Z_O)$ | $0.08$ |
| Error $(y-\hat{y})$ | $0.92$ |
| Squared error | $0.4232$ |

My network's prediction is nowhere close to the target ($\hat{y}=0.08$ vs. $y=1$), but that actually makes sense — these weights are just random starting values, not values that have been trained yet. This is exactly why backward propagation exists: in Laboratory Task 3, I'll use this same big error to figure out how to nudge the weights so $\hat{y}$ moves closer to $1$.
